# Gold: clusters de perfil por engajamento (`governor_profile_clusters_engagement`)

ADR [0022](../../docs/adr/0022-notebooks-de-diagnostico-medallion-separados-da-narrativa-do-tcc.md)
(issue [#128](https://github.com/Vini0606/Tecnicas-de-Ciencia-de-Dados-em-dados-do-Instagram/issues/128)).
Diagnóstico de `governor_profile_clusters_engagement` -- clusterização **por perfil** conforme
padrão de engajamento (`src/modeling/profile_clustering.py`, escrita via
`ModelEnricher.write_profile_clusters_engagement`), granularidade de **um governador por linha**.
Não confundir com `governor_clusters` (`gold_clusters.ipynb`), que clusteriza posts/reels
individuais por conteúdo -- espaços de cluster diferentes, mesmo `cluster_label`/`cluster_algo`
como nomes de coluna.

Notebook estritamente leitura via `DeltaRepository`, mesmo princípio da ADR
[0003](../../docs/adr/0003-desacoplar-modelagem-do-notebook-via-scripts-cli-com-checkpoint.md).

**Nota**: esta tabela não tinha dado local no ambiente em que este notebook foi escrito -- a célula
de carga abaixo levanta um erro explícito nesse caso, em vez de seguir com um DataFrame vazio
disfarçado de "sem achados".

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

from config import settings
from src.repositories.delta_repository import DeltaRepository
from src.analysis.medallion_diagnostics import (
    completeness_summary,
    count_duplicate_rows,
    with_governor_metadata,
)

load_dotenv()
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

repo = DeltaRepository(gold_dir=settings.GOLD_DIR, silver_dir=settings.SILVER_DIR)

## 1. Carga + schema

In [ ]:
try:
    df_profile_clusters = repo.load_profile_clusters_engagement()
except FileNotFoundError as e:
    raise RuntimeError(
        'governor_profile_clusters_engagement sem dado local -- rode '
        'scripts/run_profile_clustering_engagement.py primeiro (lê Silver via DeltaRepository, '
        'escreve esta tabela). Detalhe original: ' + str(e)
    ) from e
df_profile_clusters.dtypes

## 2. Completude

Um governador por linha -- duplicata por `inputUrl` seria um bug de escrita, não uma
possibilidade normal.

In [ ]:
completude = completeness_summary(df_profile_clusters)
print(f"linhas duplicadas (por inputUrl): {count_duplicate_rows(df_profile_clusters, subset=['inputUrl'])}")
completude[completude['n_nulos'] > 0]

## 3. Distribuição: tamanho de cada cluster de perfil

In [ ]:
df_profile_clusters['cluster_label'].value_counts().sort_index()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df_profile_clusters['cluster_score'], kde=False, ax=ax)
ax.set_title('Distribuição de cluster_score')
plt.tight_layout()
plt.show()

## 4. Evolução temporal

Sem tabela `_history` irmã para `governor_profile_clusters_engagement` hoje -- seção do esqueleto
padrão pulada de propósito.

## 5. Relação com covariáveis (partido/UF)

Esta tabela já tem `inputUrl` diretamente (diferente de `governor_clusters`), então o join é direto,
sem ponte via `profiles_clean`.

In [ ]:
df_com_metadado = with_governor_metadata(df_profile_clusters, repo.load_governors_metadata())
pd.crosstab(df_com_metadado['partido'], df_com_metadado['cluster_label'], normalize='index').mul(100).round(1)

## 6. Outliers

Perfis em `cluster_label == -1` (ruído do algoritmo de cluster de perfil).

In [ ]:
atipicos = df_com_metadado[df_com_metadado['cluster_label'] == -1]
print(f'{len(atipicos)} perfis em cluster_label == -1')
atipicos[['inputUrl', 'nome', 'partido', 'uf', 'cluster_algo', 'cluster_score']]

## Nota de interpretação

Cruzar `cluster_label` desta tabela (padrão de engajamento do PERFIL inteiro) com os clusters de
`gold_clusters.ipynb` (conteúdo de posts individuais) responde se perfis com engajamento parecido
também produzem o mesmo tipo de conteúdo, ou se são fenômenos independentes -- pergunta que nenhuma
das duas tabelas sozinha responde.